# Project interactive visualization
Using a dataset that your group is consider using for the term project, let's build some meaningful user-driven data visualization. Depending on your dataset this could include:
- Usage of geospatial information
- Building interactive views with widgets
- Organize multiple components into a simple dashboard

Keep these design principles in mind:
While you navigate through this notebook some things to take into consideration:
* Do not add interaction just to add it. Make sure it helps answer a question.
* Use meaningful titles and labels
* Document your code so it's readable and clean. If something does not work, document the issue and explain your best attempt.

In [75]:
## These are most likely the libraries you will use
# Add or remove imports as needed
!pip install geopy
import pandas as pd
import numpy as np

# Visualization libraries
import plotly.express as px
import plotly.graph_objects as go

# Geospatial / geocoding
from geopy.geocoders import Nominatim
import ast
# Panel
import panel as pn
pn.extension('plotly')
!pip install jupyter_bokeh
import time


[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip



[notice] A new release of pip is available: 24.3.1 -> 26.0.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [76]:
### Load your dataset here

# Example:
# df = pd.read_csv("your_dataset.csv")
df = pd.read_csv("movies.csv")

In [77]:
# Write your code here
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 4803 entries, 0 to 4802
Data columns (total 24 columns):
 #   Column                Non-Null Count  Dtype  
---  ------                --------------  -----  
 0   index                 4803 non-null   int64  
 1   budget                4803 non-null   int64  
 2   genres                4775 non-null   object 
 3   homepage              1712 non-null   object 
 4   id                    4803 non-null   int64  
 5   keywords              4391 non-null   object 
 6   original_language     4803 non-null   object 
 7   original_title        4803 non-null   object 
 8   overview              4800 non-null   object 
 9   popularity            4803 non-null   float64
 10  production_companies  4803 non-null   object 
 11  production_countries  4803 non-null   object 
 12  release_date          4802 non-null   object 
 13  revenue               4803 non-null   int64  
 14  runtime               4801 non-null   float64
 15  spoken_languages     

## Question 1: Analytical Question
Write a question about your data that can be explored with an interactive plot. A good example would be a question where you can compare different categories. Explain how the plot help answer your question and why you choose that plot type.

Examples:
- Which regions have the highest average values?
- How does one variable compare across time?

Question: Are certain genres more likely to produce profitable films?

In [78]:
# Your code . . .

df["genres"] = df["genres"].fillna("")
df["main_genre"] = df["genres"].apply(lambda x: x.split()[0] if x != "" else None)
df["profit"] = df["revenue"]-df["budget"]

genre_profit=df.groupby("main_genre",as_index=False)["profit"].mean()

fig =px.bar(
    genre_profit.sort_values("profit", ascending=False),
    x="main_genre",
    y="profit",
    title="Average Profit by Main Genre",
    labels={"main_genre": "genre", "profit":"Average Profit"},
    hover_data=["profit"]
)
fig.show()

## Question 2. Create a simple interaction plot
Create a plot, it can be related to your question #1 or a new question, that users can interact with in some meaningful way. Explain in a markdown, how does the interaction add to the analysis.

Example of possible interactions:
*   Hover over information
*   Toogle between groups

Question: Does movie popularity correlate with financial success?

In [79]:
# Your code . . .
fig =px.scatter(
    df,
    x="popularity",
    y="profit",
    title="Popularity vs Financial Success",
    labels={"popularity": "Popularity", "profit":"Average Profit"},
    hover_data=["title", "vote_average", "budget"]
)
fig.show()

## Question 3. Choropleth Planning
Design a choropleth idea using your dataset.
In a markdown cell:
*  Identify the geographic unit you would map (state, county, country, ZIP code, etc.)
*  Identify the variable you would color by
*  Explain if any aggregation or preprocessing is needed
*  Briefly describe what GeoJSON file would be required

You do not need to have a perfect dataset for this question. However, your plan should be realistic.

If your data does not fully support a choropleth, build a prototype table that explains that structure you would need before mapping.

For this chorpleth design I would create a choropleth map with a country geographic unit. Each country would represent a geographic unit, allowing us to analyze how movie performance varies globally. The variable used for coloring would be the average movie profit for each country. This would highlight which countries tend to produce more financially successful films. To prepare the data, preprocessing is needed. The production_countries column needs to be parsed to extract country names. The dataset would also be expanded since some movies are produced in more than 1 country. The date would then be grouped by country and aggregated to compute the average profit per country.

The GeoJSON file containing world countries is required and the country names in the dataset would need to match those in the file.

This choropleth would allow users to visually compare global movie performance and identify where more profitable films are produced.

## Question 4. Geospatial Possibility Check

Determine whether your dataset can support a map-based visualization.

In a markdown cell, answer one of the following:
- If **yes**, explain what geographic fields you have and what type of map is appropriate.
- If **no**, explain what is missing and what you would need to create a map.

Write code that inspects or prepares the geographic column(s) you may use.

In [80]:
# Your code . . .
df["production_countries"].head()

0    [{"iso_3166_1": "US", "name": "United States o...
1    [{"iso_3166_1": "US", "name": "United States o...
2    [{"iso_3166_1": "GB", "name": "United Kingdom"...
3    [{"iso_3166_1": "US", "name": "United States o...
4    [{"iso_3166_1": "US", "name": "United States o...
Name: production_countries, dtype: object

In [81]:
df["production_countries"].isna().sum()

np.int64(0)

In [82]:
def get_country(x):
  if isinstance(x, str) and "name" in x:
    try:
      return x.split('"name: "')[1].split('"')[0]
    except:
      return None
  return None

df["country"]=df["production_countries"].apply(get_country)
df[["production_countries", "country"]].head()

,production_countries,country
0,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",None
1,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",None
2,"[{""iso_3166_1"": ""GB"", ""name"": ""United Kingdom""...",None
3,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",None
4,"[{""iso_3166_1"": ""US"", ""name"": ""United States o...",None


## Question 5. Geopy / Location Preparation

If your dataset has location names, addresses, cities, states, or countries, use geopy on a sample of your data to geocode locations or validate location information.

If your dataset does not have data that contains location, pick 5 destination you want to visit and use geopy to geocode locations.

In [83]:

def extract_country(x):
    try:
        ## convert it into a python object 
        countries = ast.literal_eval(x)
        ## check if the list has a country and return the first country name or none if not
        return countries[0]["name"] if len(countries) > 0 else None
    except:
        return None

df["clean_country"] = df["production_countries"].apply(extract_country)
geolocator = Nominatim(user_agent="movie_app")

# Define a dictionary containing  data if data is messy 
sample_countries = df["clean_country"].dropna().unique()[:5]
## Declare an empty list to store locations of country
locations = []

for c in sample_countries:
    try:
        print("Geocoding:", c)
        loc = geolocator.geocode(c)
         ## country and coordinates returned from function stored into lists
        if loc:
            locations.append({
                "country": c,
                "latitude": loc.latitude,
                "longitude": loc.longitude
            })
        else:
            locations.append({
                "country": c,
                "latitude": None,
                "longitude": None
            })
        time.sleep(1)
    except Exception as e:
        print("Error:", e)
        locations.append({
            "country": c,
            "latitude": None,
            "longitude": None
        })
## store into dataframe
geo_df = pd.DataFrame(locations)

geo_df

Geocoding: United States of America
Geocoding: United Kingdom
Geocoding: Jamaica
Geocoding: Czech Republic
Geocoding: New Zealand


,country,latitude,longitude
0,United States of America,39.783730,-100.445882
1,United Kingdom,54.702354,-3.276575
2,Jamaica,18.185051,-77.394769
3,Czech Republic,49.743905,15.338106
4,New Zealand,-41.500083,172.834408


## Question 6. Panel Widget

Create a Panel Widget that controls something in your analysis such as the ability to choose a column, category, year, etc.

The widget should affect an output such as a plot, table, or summary statistic.

In [84]:
## for displaying dataframes
pn.extension("tabulator")
# convert to string and split at comma to get  first genre
df["main_genre"] = df["genres"].apply(
    lambda x: str(x).split(',')[0].strip() if pd.notna(x) and str(x) != '[]' else ''
)

# Make budget and revenue numeric
df["budget"] = pd.to_numeric(df["budget"], errors='coerce').fillna(0)
df["revenue"] = pd.to_numeric(df["revenue"], errors='coerce').fillna(0)

# Create profit column
df["profit"] = df["revenue"] - df["budget"]

# Panel widget
genre_select = pn.widgets.Select(
    name="Select Genre",
    options=sorted(df["main_genre"].dropna().unique().tolist())
)

# Function to show filtered table
def show_table(genre):
    temp = df[df["main_genre"] == genre][["title", "popularity", "profit"]].head(10)
    return temp

interactive_table = pn.bind(show_table, genre_select)

output_panel = pn.Column(
    genre_select,
    interactive_table
)

output_panel

BokehModel(combine_events=True, render_bundle={'docs_json': {'b54c29d4-550b-487a-a46d-cc952996569f': {'version…

## Question 7. Mini Dashboard

Build a small Panel dashboard with at least two components. Arrange the components so that it is in a readable layout. Your dashboard should include:
* At least one plot,
* An additional element of your choice such as a widget, table, second plot, etc.

In [85]:
# Write your code here

pn.extension("plotly", "tabulator")

df["main_genre"] = df["main_genre"].astype(str).str.strip()

genre_select = pn.widgets.Select(
    name="Select Genre",
    options=sorted(df["main_genre"].dropna().unique().tolist())
)

def make_plot(genre):
    temp = df[df["main_genre"] == genre].head(50)
    fig = px.scatter(
        temp,
        x="popularity",
        y="profit",
        hover_data=["title"],
        title=f"Popularity vs Profit for {genre}"
    )
    return fig

def make_table(genre):
    temp = df[df["main_genre"] == genre][["title", "popularity", "profit"]].head(10)
    return temp

plot_panel = pn.bind(make_plot, genre_select)
table_panel = pn.bind(make_table, genre_select)

q7_dashboard = pn.Column(
    "## Mini Dashboard",
    genre_select,
    pn.Row(
        plot_panel,
        table_panel
    )
)

q7_dashboard


BokehModel(combine_events=True, render_bundle={'docs_json': {'5c189ce9-234c-4341-a6f0-d03acaf423f6': {'version…

## Question 8. Reflection

Write a short reflection addressing all of the following:
- Which interactive element was most useful in your notebook?
- What was the hardest part of working with your dataset?
- Did implementing interactive tools help your analysis? Why or why not?
- If you had more time, what would you improve or add?

## Question 8. Reflection

Write a short reflection addressing all of the following:
- Which interactive element was most useful in your notebook?
The most useful interactive element from this notebook was the select as it allowed for the data to be filtered based on the genre which made it easier to visualize the popularity of certain movies in certain genres such as comedy or crime. It allowed us to see profit and popularity based on the type of movie without us manually checking what type of movie and then looking for other movies of the same type among different categories. 
- What was the hardest part of working with your dataset?
The hardest part was ensuring that the data was clean so it an be used as we had to fix the columns to use them such as by making sure they were numeric as if they were not then we were unable to filter by genere.Overall, formatting and missing value lead to the plots not displaying correctly which we had to fix. 
- Did implementing interactive tools help your analysis? Why or why not?
Yes, it definitely helped with analysis as they made it easier to notice patterns and comapre different generes in an easier manner. We did not have to create a bunch of plots as we were able to simple filter using the select tool. It made comparisons faster. 
- If you had more time, what would you improve or add?
We would add more filters such as filtering by the year or if a movie had several generes then we would need more select options. If we had more times we would add more graphs such as a bar graph for top genres by profit or more calcuations of the average profit and max profit of a genre. All of this would help create a stronger analysis of the data. 